In [ ]:
import pandas as pd
import numpy as np

# Configuration: File paths
INPUT_FILE = 'LFB Incident data from 2024 onwards.xlsx'
OUTPUT_STATIONS = 'stations.csv'
OUTPUT_TYPES = 'incident_types.csv'
OUTPUT_INCIDENTS = 'incidents.csv'

def clean_and_normalize_data():
    print("Loading raw Excel file... (This may take a moment)")
    df = pd.read_excel(INPUT_FILE)

    # --- 1. Data Cleaning ---
    # Convert Excel timestamps to MySQL-friendly Date format (YYYY-MM-DD)
    df['DateOfCall'] = pd.to_datetime(df['DateOfCall']).dt.strftime('%Y-%m-%d')
    
    # Ensure Time is string format
    df['TimeOfCall'] = df['TimeOfCall'].astype(str)

    # Drop rows where critical keys are missing
    df = df.dropna(subset=['IncidentNumber', 'IncidentStationGround'])

    # --- 2. Normalization: Create 'Station' Table ---
    # Extracts unique stations and their boroughs to remove partial dependency
    stations = df[['IncidentStationGround', 'IncGeo_BoroughName']].drop_duplicates(subset=['IncidentStationGround'])
    stations = stations.rename(columns={
        'IncidentStationGround': 'StationName',
        'IncGeo_BoroughName': 'Borough'
    })
    stations.to_csv(OUTPUT_STATIONS, index=False)
    print(f"-> Exported {OUTPUT_STATIONS} (Station Lookup)")

    # --- 3. Normalization: Create 'IncidentType' Table ---
    # Creates unique IDs for incident categories to remove redundancy
    types = df[['IncidentGroup', 'StopCodeDescription']].drop_duplicates()
    types = types.reset_index(drop=True)
    types['TypeID'] = types.index + 1  # Generate IDs starting from 1
    
    # Save the lookup table
    types.to_csv(OUTPUT_TYPES, index=False)
    print(f"-> Exported {OUTPUT_TYPES} (Type Lookup)")

    # --- 4. Create Main 'Incident' Table ---
    # Merge the new TypeID back into the main dataframe
    df = df.merge(types, on=['IncidentGroup', 'StopCodeDescription'], how='left')

    # Select only columns required for the Schema
    incidents = df[[
        'IncidentNumber',
        'DateOfCall',
        'TimeOfCall',
        'TypeID',                # Foreign Key
        'IncidentStationGround', # Foreign Key
        'Latitude',
        'Longitude',
        'FirstPumpArriving_AttendanceTime',
        'NumPumpsAttending'
    ]]

    # Rename columns to match MySQL Schema exactly
    incidents = incidents.rename(columns={
        'IncidentStationGround': 'StationName',
        'FirstPumpArriving_AttendanceTime': 'ResponseTimeSeconds',
        'NumPumpsAttending': 'PumpCount'
    })

    # Handle Missing Values for SQL Import
    # We replace NaNs with the string 'NULL' so SQL LOAD DATA handles them correctly
    cols_to_clean = ['Latitude', 'Longitude', 'ResponseTimeSeconds', 'PumpCount']
    for col in cols_to_clean:
        incidents[col] = incidents[col].fillna('NULL')

    # Save the final transactional table
    incidents.to_csv(OUTPUT_INCIDENTS, index=False)
    print(f"-> Exported {OUTPUT_INCIDENTS} (Main Data Table)")
    print("Preprocessing Complete.")

if __name__ == "__main__":
    clean_and_normalize_data()

Loading data... this may take a moment.
-> Created stations.csv
-> Created incident_types.csv
-> Created incidents.csv
Done! Download the 3 CSV files from the files panel.
